# Rastreamento de Objetos com YOLO + ByteTrack

Neste notebook, iremos explorar o rastreamento de objetos (*object tracking*), combinando os modelos de detecção da família YOLO com o algoritmo `ByteTrack`, através da biblioteca `ultralytics`.

A detecção de objetos identifica "o quê" e "onde" em cada quadro, isoladamente. O rastreamento vai além: atribui um identificador (ID) único e persistente para cada objeto, mantendo esse ID ao longo dos quadros do vídeo, mesmo quando o objeto muda de posição ou fica parcialmente encoberto por instantes. É a base de aplicações de hiperautomação como contagem de veículos e pessoas, análise de fluxo de tráfego e monitoramento de pátios e linhas de produção.

In [ ]:
from IPython import get_ipython
if 'google.colab' in str(get_ipython()):
    print("Preparando ambiente Google Colab")
    !pip install opencv-python==5.0.0.93 
    !pip install opencv-contrib-python==5.0.0.93
    !pip install ultralytics
    !pip install lap
    !pip install moviepy==2.2.1
    !git clone https://github.com/pvoloshyn/curso-visao-computacional.git
    %cd curso-visao-computacional
else:
    pass

## Carregando bibliotecas

Além das bibliotecas que já usamos em outros notebooks, vamos utilizar mais duas.
* `ultralytics`: Biblioteca que facilita o acesso e uso dos modelos da família YOLO, incluindo o rastreamento com `ByteTrack`.
* `moviepy`: Biblioteca para editar e apresentar vídeos.

In [ ]:
from ultralytics import YOLO

from collections import defaultdict, deque
from pathlib import Path
from moviepy import VideoFileClip

import cv2
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
# Indica ao notebook to render figures in-page.
%matplotlib inline  
from IPython.display import Image, Video

## 1. Como Funciona o Rastreamento

O `ByteTrack` é um algoritmo de rastreamento *multi-objeto* que funciona em cima das caixas detectadas pelo YOLO em cada quadro, sem precisar de nenhum treinamento adicional. Resumidamente, para cada novo quadro, ele:

1. Usa um filtro de Kalman para prever, com base no movimento anterior, onde cada objeto já rastreado deve estar no quadro atual.
2. Associa as novas detecções do YOLO às previsões através da sobreposição das caixas (IoU), mantendo o mesmo ID quando a associação é boa.
3. Cria um novo ID para detecções de alta confiança que não foram associadas a nenhum rastro existente.

O grande diferencial do `ByteTrack` em relação a rastreadores mais simples é não descartar de cara as detecções de baixa confiança: elas também são usadas para tentar recuperar rastros existentes antes de serem descartadas. Isso reduz bastante a troca de IDs quando um objeto fica parcialmente encoberto ou borrado por alguns quadros.

## 2. Rastreamento em um Quadro

Antes de processar o vídeo inteiro, vamos entender a API em um único quadro.

### 2.1. Carregando o modelo

Assim como na detecção de objetos, vamos usar a versão 26 do YOLO.

In [ ]:
# Caso tenha problemas em usar a versão 26, descomente a linha de baixo e comente a outra
#detect_model = YOLO('modelos/yolov8n.pt')
detect_model = YOLO('modelos/yolo26n.pt')

### 2.2. Carregando um quadro do vídeo

Vamos usar o `cv2.VideoCapture` para abrir o vídeo e capturar um único quadro.

In [ ]:
video_path = 'imagens/05/2099406-hd_1920_1080_30fps.mp4'

cap = cv2.VideoCapture(video_path)
ok, quadro = cap.read()
cap.release()

quadro_rgb = cv2.cvtColor(quadro, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(10, 8))
plt.imshow(quadro_rgb)
plt.axis('off')
plt.show()

### 2.3. Rastreamento com o ByteTrack

Ao invés de chamar o modelo diretamente, usamos o método `track()`. Por padrão, a `ultralytics` usa o `BoT-SORT` como rastreador; para usar o `ByteTrack`, indicamos `tracker='bytetrack.yaml'`.

In [ ]:
results = detect_model.track(quadro, tracker='bytetrack.yaml')

### 2.4. Apresentando e Interpretando Resultados

O resultado continua sendo um objeto `Results`, com a propriedade `boxes` de sempre. A diferença é que agora `boxes.id` traz o identificador de rastreamento de cada objeto (`None` quando ainda não há nenhum objeto sendo rastreado).

In [ ]:
for box in results[0].boxes:
    track_id = int(box.id[0])
    classe = int(box.cls[0])
    confianca = float(box.conf[0])

    print(
        track_id,
        results[0].names[classe],
        round(confianca, 2),
        box.xyxy[0].round().tolist()
    )

O método `plot()` também já apresenta o ID de rastreamento junto da classe e da confiança.

In [ ]:
result_img = cv2.cvtColor(results[0].plot(), cv2.COLOR_BGR2RGB)

plt.figure(figsize=(12, 8))
plt.imshow(result_img)
plt.axis('off')
plt.show()

## 3. Rastreamento no Vídeo Completo

Em um único quadro, o rastreamento não faz muita diferença perto da detecção comum: cada ID novo poderia muito bem ser o índice da caixa. É processando o vídeo, quadro a quadro, que ele mostra sua utilidade, mantendo o mesmo ID para cada carro, moto, ciclista e pedestre ao longo do tempo.

### 3.1. Carregando o vídeo

Assim como fizemos no notebook anterior, usamos o `cv2.VideoCapture` para abrir o vídeo e ler suas propriedades.

Recarregamos também o `detect_model`: na seção 2 chamamos `track()` uma vez sem `persist=True` (para o exemplo de um único quadro), e essa chamada deixa esse objeto YOLO com o rastreador "contaminado". Reaproveitá-lo faz com que, mesmo passando `persist=True` depois, ele reinicie o `ByteTrack` do zero em todo quadro, embaralhando os IDs. Um modelo recém-carregado evita esse problema.

In [ ]:
detect_model = YOLO('modelos/yolo26n.pt')

output_path = 'output/transito_rastreado.mp4'

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
largura = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
altura = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total_quadros = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

print(f"Resolução: {largura}x{altura}, {fps:.1f} fps, {total_quadros} quadros")

### 3.2. Função de Apresentação

Antes de processar o vídeo, vamos criar uma função para desenhar, em cada quadro, a caixa, o ID, a classe e a confiança de cada objeto rastreado, além de um rastro com as últimas posições do centro de cada objeto, o que deixa bem visível o caminho percorrido por ele.

Para isso, mantemos duas estruturas entre os quadros:
* `trajetorias`: um dicionário que guarda, para cada ID, uma fila com as últimas posições do centro do objeto, junto do índice do quadro em que cada posição foi vista.
* `cores`: um dicionário que fixa uma cor por ID, para o rastro e a caixa não ficarem trocando de cor a cada quadro.

Guardamos o índice do quadro junto de cada posição porque o `ByteTrack` pode perder um objeto por alguns quadros (oclusão, baixa confiança) e recuperá-lo mais tarde com o mesmo ID, possivelmente em outro ponto da imagem. Sem essa informação, o rastro ligaria a última posição antes da perda diretamente à posição de recuperação, desenhando uma linha reta atravessando a imagem. Por isso só ligamos dois pontos do rastro quando eles vêm de quadros consecutivos.

In [ ]:
def desenhar_rastreamento(img: np.ndarray, result, trajetorias: dict, cores: dict, quadro_idx: int) -> np.ndarray:
    """ Desenha caixa, ID, classe/confiança e rastro de cada objeto rastreado """
    img = img.copy()

    boxes = result.boxes
    if boxes.id is None:
        return img

    ids = boxes.id.int().cpu().tolist()
    classes = boxes.cls.int().cpu().tolist()
    confiancas = boxes.conf.cpu().tolist()
    caixas = boxes.xyxy.cpu().numpy()

    for track_id, classe, confianca, (x1, y1, x2, y2) in zip(ids, classes, confiancas, caixas):
        # Fixa uma cor para o ID, gerada de forma determinística a partir dele
        if track_id not in cores:
            cores[track_id] = tuple(int(c) for c in np.random.default_rng(track_id).integers(60, 255, size=3))
        cor = cores[track_id]

        # Guarda a posição atual do centro do objeto no seu rastro, junto do quadro em que foi vista
        centro = (int((x1 + x2) / 2), int((y1 + y2) / 2))
        trajetorias[track_id].append((quadro_idx, centro))

        # Desenha o rastro, ligando as últimas posições do centro — mas só entre quadros consecutivos,
        # para não desenhar uma linha reta atravessando a imagem quando o ByteTrack recupera o ID
        # de um objeto depois de tê-lo perdido por alguns quadros
        pontos = list(trajetorias[track_id])
        for (quadro1, p1), (quadro2, p2) in zip(pontos, pontos[1:]):
            if quadro2 - quadro1 == 1:
                cv2.line(img, p1, p2, cor, 2)

        # Desenha a caixa e o rótulo
        label = f"#{track_id} {result.names[classe]} {confianca:.2f}"

        cv2.rectangle(img, (int(x1), int(y1)), (int(x2), int(y2)), cor, 2)
        cv2.putText(
            img,
            label,
            (int(x1), max(int(y1) - 8, 0)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.6,
            cor,
            2
        )

    return img

### 3.3. Processando o vídeo, quadro a quadro

Para cada quadro, chamamos `track()` com `persist=True`. Esse parâmetro é essencial: ele diz à `ultralytics` para manter o estado do rastreador entre uma chamada e outra, associando as detecções do quadro atual às dos quadros anteriores. Sem ele, um rastreador novo seria criado a cada chamada, e todo objeto ganharia um ID novo em todo quadro.

Aproveitamos o mesmo laço para contar, em `contagem`, quantos IDs únicos de cada classe passaram pelo vídeo, um exemplo simples de métrica de negócio (quantos carros, pessoas, motos etc. passaram pela via) que só é possível calcular porque estamos rastreando, e não apenas detectando.

Como o vídeo tem quase mil quadros em alta resolução, essa célula pode levar alguns minutos para rodar em CPU.

In [ ]:
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(output_path, fourcc, fps, (largura, altura))

trajetorias = defaultdict(lambda: deque(maxlen=30))
cores = {}
contagem = defaultdict(set)

quadros_amostra = []
intervalo_amostra = max(total_quadros // 4, 1)

quadro_idx = 0
while True:
    ok, quadro = cap.read()
    if not ok:
        break

    results = detect_model.track(quadro, persist=True, tracker='bytetrack.yaml', verbose=False)
    result = results[0]

    if result.boxes.id is not None:
        ids = result.boxes.id.int().cpu().tolist()
        classes = result.boxes.cls.int().cpu().tolist()
        for track_id, classe in zip(ids, classes):
            contagem[result.names[classe]].add(track_id)

    quadro_anotado = desenhar_rastreamento(quadro, result, trajetorias, cores, quadro_idx)
    writer.write(quadro_anotado)

    if quadro_idx % intervalo_amostra == 0:
        quadros_amostra.append(cv2.cvtColor(quadro_anotado, cv2.COLOR_BGR2RGB))

    quadro_idx += 1

cap.release()
writer.release()

print(f"Vídeo processado salvo em '{output_path}' ({quadro_idx} quadros).")

### 3.4. Apresentando o Resultado

Assim como no notebook anterior, primeiro conferimos uma amostra dos quadros processados com o `matplotlib`, e depois tentamos exibir o vídeo completo.

In [ ]:
fig, eixos = plt.subplots(1, len(quadros_amostra), figsize=(5 * len(quadros_amostra), 5))

for eixo, quadro in zip(eixos, quadros_amostra):
    eixo.imshow(quadro)
    eixo.axis('off')

plt.tight_layout()
plt.show()

Se o seu navegador suportar o codec do vídeo gerado, ele será reproduzido logo abaixo. Caso contrário, abra o arquivo `output/transito_rastreado.mp4` diretamente em um player de vídeo.

In [ ]:
def display_video(filename: str, **kwargs) -> None:
    """ Apresenta vídeo no notebook """
    clip = VideoFileClip(filename)
    html_embed = clip.display_in_notebook(**kwargs)
    # O moviepy cria um arquivo temporário. Só vamos apagar ele.
    Path("__temp__.mp4").unlink(missing_ok=True)
    return html_embed

In [ ]:
display_video(output_path, loop=1, width=800)

## 4. Contagem de Objetos Únicos

Como cada objeto manteve o mesmo ID do início ao fim de sua passagem pelo vídeo, o tamanho de cada conjunto em `contagem` nos dá o número de objetos únicos rastreados por classe. Bem diferente da soma de detecções por quadro, que contaria o mesmo carro dezenas de vezes.

In [ ]:
for classe, ids in sorted(contagem.items(), key=lambda item: -len(item[1])):
    print(f"{classe}: {len(ids)} objeto(s) único(s)")